# Module 3 Homework – Stock Markets Analytics Zoomcamp

**Note:** the notebook first tries to download the course dataset. If there is no internet access (as in the environment where this file was generated), it falls back to a **synthetic stand-in dataset** with the same column names so every cell runs. The numbers saved in the outputs below therefore come from synthetic data. **Re-run all cells in Colab (with internet) to get your real homework answers.**

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

USING_SYNTHETIC = False
URL = "https://raw.githubusercontent.com/DataTalksClub/stock-markets-analytics-zoomcamp/main/03-model/data/module3_dataset.csv"

try:
    df = pd.read_csv(URL)
    print("Loaded real dataset:", df.shape)
except Exception as e:
    USING_SYNTHETIC = True
    print("Could not download dataset -> using SYNTHETIC data (answers are NOT real).")
    rng = np.random.default_rng(42)
    dates = pd.bdate_range("2015-01-01", "2025-06-30")
    tickers = {"AAPL": "stock", "MSFT": "stock", "JPM": "stock", "SPY": "ETF", "QQQ": "ETF"}
    rows = []
    for t, tt in tickers.items():
        n = len(dates)
        d = pd.DataFrame({"Date": dates, "Ticker": t, "ticker_type": tt})
        d["DGS10"] = np.clip(2.5 + np.cumsum(rng.normal(0, 0.03, n)), 0.5, 5.5)
        d["DGS5"] = np.clip(d["DGS10"] - rng.normal(0.2, 0.1, n), 0.3, 5.5)
        d["FEDFUNDS"] = np.clip(d["DGS10"] - rng.normal(0.5, 0.3, n), 0.1, 5.5)
        d["rsi"] = rng.uniform(20, 80, n)
        d["mom_30"] = rng.normal(0, 0.08, n)
        d["growth_future_30d"] = 0.3 * d["mom_30"] + rng.normal(0.01, 0.07, n)
        d["is_positive_growth_30d_future"] = (d["growth_future_30d"] > 0).astype(int)
        rows.append(d)
    df = pd.concat(rows, ignore_index=True)
    df["Month"] = df["Date"].dt.strftime("%B")
    df["Weekday"] = df["Date"].dt.weekday
    df["split"] = np.where(df["Date"] < "2022-01-01", "train",
                   np.where(df["Date"] < "2023-07-01", "val", "test"))
    df["pred0"] = (df["rsi"] < 40).astype(int)
    df["pred1"] = (df["mom_30"] > 0).astype(int)
    df["pred2"] = (df["DGS10"] < 3).astype(int)
    print("Synthetic shape:", df.shape)

new_df = df.copy()
new_df["Date"] = pd.to_datetime(new_df["Date"])
new_df.head()

Could not download dataset -> using SYNTHETIC data (answers are NOT real).
Synthetic shape: (13690, 16)


,Date,Ticker,ticker_type,DGS10,DGS5,FEDFUNDS,rsi,mom_30,growth_future_30d,is_positive_growth_30d_future,Month,Weekday,split,pred0,pred1,pred2
0,2015-01-01,AAPL,stock,2.509142,2.063317,2.273630,69.662894,0.000710,0.011166,1,January,3,train,0,1,1
1,2015-01-02,AAPL,stock,2.477942,2.138735,1.787319,76.588658,0.087516,-0.028180,0,January,4,train,0,1,1
2,2015-01-05,AAPL,stock,2.500456,2.385749,1.734449,71.409340,-0.047854,-0.036432,0,January,0,train,0,0,1
3,2015-01-06,AAPL,stock,2.528672,2.263915,2.324369,40.034813,-0.020752,0.138772,1,January,1,train,0,0,1
4,2015-01-07,AAPL,stock,2.470141,2.353294,1.768957,25.611809,0.001004,0.056808,1,January,2,train,1,1,1


## Question 1 – Month + week-of-month dummies

In [2]:
new_df['week_of_month'] = (new_df['Date'].dt.day - 1) // 7 + 1
new_df['month_wom'] = new_df['Date'].dt.strftime('%B') + '_w' + new_df['week_of_month'].astype(str)

cat_features = ['Month', 'Weekday', 'Ticker', 'ticker_type', 'month_wom']
df_dummies = pd.get_dummies(new_df[cat_features].astype(str), drop_first=False)
df_dummies['is_positive_growth_30d_future'] = new_df['is_positive_growth_30d_future']

correlations = df_dummies.astype(float).corr()['is_positive_growth_30d_future']
wom_corrs = correlations[correlations.index.str.startswith('month_wom_')].to_frame(name='corr')
wom_corrs['abs_corr'] = wom_corrs['corr'].abs()
wom_corrs_sorted = wom_corrs.sort_values('abs_corr', ascending=False)

q1_answer_feature = wom_corrs_sorted.index[0]
q1_answer_val = round(wom_corrs_sorted['abs_corr'].iloc[0], 3)
print(f"Top Month-WOM Dummy: {q1_answer_feature}")
print(f"Answer Q1 - Highest absolute correlation: {q1_answer_val:.3f}")

Top Month-WOM Dummy: month_wom_June_w2
Answer Q1 - Highest absolute correlation: 0.021


## Question 2 – New hand-written rules (precision on TEST)

In [3]:
new_df['pred3_manual_dgs10_5'] = ((new_df['DGS10'] <= 4.5) & (new_df['DGS5'] <= 4)).astype(int)
new_df['pred4_manual_dgs10_fedfunds'] = ((new_df['DGS10'] > 4) & (new_df['FEDFUNDS'] <= 4.795)).astype(int)

test_mask = new_df['split'] == 'test'
test_df = new_df[test_mask]

y_test = test_df['is_positive_growth_30d_future']
p3 = precision_score(y_test, test_df['pred3_manual_dgs10_5'], zero_division=0)
p4 = precision_score(y_test, test_df['pred4_manual_dgs10_fedfunds'], zero_division=0)

print(f"Precision pred3: {p3:.4f}")
print(f"Precision pred4: {p4:.4f}")
print(f"Answer Q2 - Best new precision: {round(max(p3, p4), 3):.3f}")

Precision pred3: 0.5451
Precision pred4: 0.5508
Answer Q2 - Best new precision: 0.551


## Question 3 – Unique correct predictions of a depth-10 decision tree

**Fix vs. the original script:** columns that contain the future outcome (`growth_future_*`) are excluded from the features. Leaving them in leaks the answer into the tree.

In [4]:
exclude = {'Date', 'Ticker', 'split', 'is_positive_growth_30d_future', 'week_of_month', 'month_wom'}
feature_cols = [c for c in new_df.columns
                if c not in exclude
                and not c.startswith('pred')
                and not c.startswith('growth_future')]

X_all = pd.get_dummies(new_df[feature_cols], drop_first=True).fillna(0)
y_all = new_df['is_positive_growth_30d_future']

train_val_mask = new_df['split'].isin(['train', 'val'])
test_mask = new_df['split'] == 'test'
X_train_val, y_train_val = X_all[train_val_mask], y_all[train_val_mask]

clf10 = DecisionTreeClassifier(max_depth=10, random_state=42)
clf10.fit(X_train_val, y_train_val)
new_df['pred5_clf_10'] = clf10.predict(X_all)

hand_preds = ['pred0', 'pred1', 'pred2', 'pred3_manual_dgs10_5', 'pred4_manual_dgs10_fedfunds']
hand_correct = pd.DataFrame({c: new_df[c] == y_all for c in hand_preds if c in new_df.columns})

pred5_correct = new_df['pred5_clf_10'] == y_all
new_df['only_pred5_is_correct'] = (pred5_correct & ~hand_correct.any(axis=1)).astype(int)

ans3 = new_df.loc[test_mask, 'only_pred5_is_correct'].sum()
print(f"Features used: {X_all.shape[1]}")
print(f"Answer Q3 - Records in TEST where only pred5 is correct: {ans3}")

Features used: 18
Answer Q3 - Records in TEST where only pred5 is correct: 58


## Question 4 – Tune tree depth (1–12)

In [5]:
best_depth, best_prec = None, -1.0
for depth in range(1, 13):
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train_val, y_train_val)
    prec = precision_score(y_all[test_mask], clf.predict(X_all[test_mask]), zero_division=0)
    print(f"Depth {depth:2d} | Test precision: {prec:.4f}")
    if prec > best_prec:
        best_prec, best_depth = prec, depth

print(f"\nAnswer Q4 - Optimal depth: {best_depth} (precision {best_prec:.4f})")

clf_best = DecisionTreeClassifier(max_depth=best_depth, random_state=42).fit(X_train_val, y_train_val)
new_df['pred6_clf_best'] = clf_best.predict(X_all)

Depth  1 | Test precision: 0.6743
Depth  2 | Test precision: 0.5915
Depth  3 | Test precision: 0.6294
Depth  4 | Test precision: 0.5886
Depth  5 | Test precision: 0.5995
Depth  6 | Test precision: 0.5916
Depth  7 | Test precision: 0.6019
Depth  8 | Test precision: 0.6007
Depth  9 | Test precision: 0.5987
Depth 10 | Test precision: 0.6053
Depth 11 | Test precision: 0.6031
Depth 12 | Test precision: 0.5999

Answer Q4 - Optimal depth: 1 (precision 0.6743)


## Question 5 – What data is missing? (exploratory)

In [6]:
q5_response = """
Proposed additional indicators & data sources:

1. Market microstructure & sentiment
   - Options indicators: implied-volatility skew, put/call ratio, VIX term structure.
   - News/earnings-call sentiment scores (LLM-derived) from headlines and transcripts.

2. Global macro & liquidity
   - Central bank balance sheets (Fed WALCL, ECB, RBI).
   - 10Y-2Y yield spread momentum and real rates (TIPS yield).

Reasoning:
DGS10 / FEDFUNDS move slowly and describe the broad regime; technical indicators describe
price momentum. Options-implied expectations and liquidity measures bridge the gap between
long-term macro trends and the 30-day horizon being predicted.
"""
print(q5_response)


Proposed additional indicators & data sources:

1. Market microstructure & sentiment
   - Options indicators: implied-volatility skew, put/call ratio, VIX term structure.
   - News/earnings-call sentiment scores (LLM-derived) from headlines and transcripts.

2. Global macro & liquidity
   - Central bank balance sheets (Fed WALCL, ECB, RBI).
   - 10Y-2Y yield spread momentum and real rates (TIPS yield).

Reasoning:
DGS10 / FEDFUNDS move slowly and describe the broad regime; technical indicators describe
price momentum. Options-implied expectations and liquidity measures bridge the gap between
long-term macro trends and the 30-day horizon being predicted.

